# Practical 11: Named Entity Recognition (NER)

This notebook demonstrates various approaches to Named Entity Recognition:
1. Using spaCy (Pre-trained)
2. Using NLTK
3. Using Transformers (BERT-based)
4. Custom NER with BiLSTM-CRF

## Part 1: Named Entity Recognition using spaCy

In [ ]:
# Install required packages (run once)
# !pip install spacy
# !python -m spacy download en_core_web_sm

In [ ]:
import spacy
from spacy import displacy

# Load pre-trained model
nlp = spacy.load("en_core_web_sm")

print("spaCy model loaded successfully!")

In [ ]:
# Sample text
text = """
Apple Inc. was founded by Steve Jobs, Steve Wozniak, and Ronald Wayne in April 1976. 
The company is headquartered in Cupertino, California. In 2023, Tim Cook serves as the CEO. 
Apple's products include the iPhone, iPad, and MacBook. The company's market value exceeded 
$3 trillion in January 2022. Microsoft and Google are major competitors in the tech industry.
"""

# Process the text
doc = nlp(text)

print("Named Entities found:\n")
print(f"{'Entity':<30} {'Label':<15} {'Description'}")
print("-" * 70)

for ent in doc.ents:
    print(f"{ent.text:<30} {ent.label_:<15} {spacy.explain(ent.label_)}")

In [ ]:
# Visualize entities (works in Jupyter)
displacy.render(doc, style="ent", jupyter=True)

In [ ]:
# Extract entities by type
def extract_entities_by_type(doc):
    entities = {}
    for ent in doc.ents:
        if ent.label_ not in entities:
            entities[ent.label_] = []
        entities[ent.label_].append(ent.text)
    return entities

entities_dict = extract_entities_by_type(doc)

print("\nEntities grouped by type:\n")
for entity_type, entity_list in entities_dict.items():
    print(f"{entity_type}: {', '.join(set(entity_list))}")

## Part 2: Named Entity Recognition using NLTK

In [ ]:
import nltk
from nltk import word_tokenize, pos_tag, ne_chunk
from nltk.tree import Tree

# Download required NLTK data (run once)
# nltk.download('punkt')
# nltk.download('averaged_perceptron_tagger')
# nltk.download('maxent_ne_chunker')
# nltk.download('words')

In [ ]:
# Sample text
text = "Barack Obama was born in Hawaii. He served as the 44th President of the United States from 2009 to 2017."

# Tokenize and POS tag
tokens = word_tokenize(text)
pos_tags = pos_tag(tokens)

print("POS Tags:")
print(pos_tags[:10])  # Show first 10

In [ ]:
# Named Entity Recognition
named_entities = ne_chunk(pos_tags)

print("\nNamed Entities (NLTK):\n")
for chunk in named_entities:
    if hasattr(chunk, 'label'):
        entity = ' '.join(c[0] for c in chunk)
        entity_type = chunk.label()
        print(f"{entity:<30} -> {entity_type}")

In [ ]:
# Function to extract named entities
def extract_entities_nltk(text):
    entities = []
    tokens = word_tokenize(text)
    pos_tags = pos_tag(tokens)
    chunks = ne_chunk(pos_tags)
    
    for chunk in chunks:
        if hasattr(chunk, 'label'):
            entity_text = ' '.join(c[0] for c in chunk)
            entity_type = chunk.label()
            entities.append((entity_text, entity_type))
    
    return entities

# Test the function
test_text = "Microsoft was founded by Bill Gates and Paul Allen in Seattle, Washington."
entities = extract_entities_nltk(test_text)

print("\nExtracted Entities:")
for entity, entity_type in entities:
    print(f"{entity} -> {entity_type}")

## Part 3: Named Entity Recognition using Transformers (BERT)

In [ ]:
# Install transformers (run once)
# !pip install transformers torch

In [ ]:
from transformers import pipeline

# Load pre-trained NER pipeline
ner_pipeline = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

print("Transformer NER model loaded!")

In [ ]:
# Sample text
text = """
Elon Musk is the CEO of Tesla and SpaceX. He was born in Pretoria, South Africa. 
Tesla's headquarters is in Austin, Texas. SpaceX successfully launched Starship in 2023.
"""

# Perform NER
results = ner_pipeline(text)

print("Named Entities (BERT):\n")
print(f"{'Entity':<30} {'Type':<15} {'Score'}")
print("-" * 60)

for entity in results:
    print(f"{entity['word']:<30} {entity['entity_group']:<15} {entity['score']:.4f}")

In [ ]:
# Function to extract and group entities
def extract_entities_transformer(text, threshold=0.9):
    results = ner_pipeline(text)
    
    entities_by_type = {}
    for entity in results:
        if entity['score'] >= threshold:
            entity_type = entity['entity_group']
            entity_text = entity['word']
            
            if entity_type not in entities_by_type:
                entities_by_type[entity_type] = []
            entities_by_type[entity_type].append(entity_text)
    
    return entities_by_type

# Test
entities = extract_entities_transformer(text, threshold=0.85)

print("\nGrouped Entities (confidence >= 0.85):\n")
for entity_type, entity_list in entities.items():
    print(f"{entity_type}: {', '.join(entity_list)}")

## Part 4: Custom NER with BiLSTM-CRF (PyTorch)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

print(f"PyTorch version: {torch.__version__}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Sample training data (BIO format)
# B-PER: Beginning of Person, I-PER: Inside Person
# B-ORG: Beginning of Organization, I-ORG: Inside Organization
# B-LOC: Beginning of Location, I-LOC: Inside Location
# O: Outside (not an entity)

training_data = [
    (["Apple", "Inc", "is", "located", "in", "Cupertino"], 
     ["B-ORG", "I-ORG", "O", "O", "O", "B-LOC"]),
    
    (["Steve", "Jobs", "founded", "Apple"], 
     ["B-PER", "I-PER", "O", "B-ORG"]),
    
    (["Microsoft", "is", "based", "in", "Seattle"], 
     ["B-ORG", "O", "O", "O", "B-LOC"]),
    
    (["Bill", "Gates", "lives", "in", "Washington"], 
     ["B-PER", "I-PER", "O", "O", "B-LOC"]),
    
    (["Google", "was", "founded", "by", "Larry", "Page"], 
     ["B-ORG", "O", "O", "O", "B-PER", "I-PER"]),
]

print("Sample training data:")
for words, tags in training_data[:2]:
    print(f"Words: {words}")
    print(f"Tags:  {tags}")
    print()

In [ ]:
# Build vocabularies
def build_vocab(training_data):
    word_to_idx = {"<PAD>": 0, "<UNK>": 1}
    tag_to_idx = {"<PAD>": 0}
    
    for words, tags in training_data:
        for word in words:
            if word not in word_to_idx:
                word_to_idx[word] = len(word_to_idx)
        
        for tag in tags:
            if tag not in tag_to_idx:
                tag_to_idx[tag] = len(tag_to_idx)
    
    idx_to_tag = {v: k for k, v in tag_to_idx.items()}
    
    return word_to_idx, tag_to_idx, idx_to_tag

word_to_idx, tag_to_idx, idx_to_tag = build_vocab(training_data)

print(f"Vocabulary size: {len(word_to_idx)}")
print(f"Tag set size: {len(tag_to_idx)}")
print(f"\nTags: {list(tag_to_idx.keys())}")

In [ ]:
# BiLSTM-CRF Model
class BiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, tag_to_idx, embedding_dim=50, hidden_dim=100):
        super(BiLSTM_CRF, self).__init__()
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.vocab_size = vocab_size
        self.tag_to_idx = tag_to_idx
        self.tagset_size = len(tag_to_idx)
        
        self.word_embeds = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim // 2,
                            num_layers=1, bidirectional=True, batch_first=True)
        
        # Maps the output of the LSTM into tag space
        self.hidden2tag = nn.Linear(hidden_dim, self.tagset_size)
        
        # CRF transitions
        self.transitions = nn.Parameter(
            torch.randn(self.tagset_size, self.tagset_size))
        
        # These two statements enforce the constraint that we never transfer
        # to the start tag and we never transfer from the stop tag
        self.transitions.data[tag_to_idx["<PAD>"], :] = -10000
        self.transitions.data[:, tag_to_idx["<PAD>"]] = -10000
    
    def forward(self, sentence):
        # Get the emission scores from the BiLSTM
        lstm_feats = self._get_lstm_features(sentence)
        
        # Find the best path, given the features
        score, tag_seq = self._viterbi_decode(lstm_feats)
        return score, tag_seq
    
    def _get_lstm_features(self, sentence):
        embeds = self.word_embeds(sentence)
        lstm_out, _ = self.lstm(embeds)
        lstm_feats = self.hidden2tag(lstm_out)
        return lstm_feats
    
    def _viterbi_decode(self, feats):
        # Simplified Viterbi for demonstration
        # In practice, use a full CRF implementation
        batch_size, seq_len, tagset_size = feats.shape
        
        # Get the best tags
        _, tags = torch.max(feats, dim=2)
        
        return 0, tags

# Initialize model
model = BiLSTM_CRF(len(word_to_idx), tag_to_idx, embedding_dim=50, hidden_dim=100)
model = model.to(device)

print("BiLSTM-CRF model initialized")
print(f"Model parameters: {sum(p.numel() for p in model.parameters())}")

In [ ]:
# Prepare data for training
def prepare_sequence(seq, to_idx):
    idxs = [to_idx.get(w, to_idx["<UNK>"]) if "<UNK>" in to_idx else to_idx.get(w, 0) for w in seq]
    return torch.tensor(idxs, dtype=torch.long)

# Simple training loop (demonstration)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=0)

print("Training model...\n")

for epoch in range(20):
    total_loss = 0
    
    for words, tags in training_data:
        model.zero_grad()
        
        # Prepare inputs
        sentence_in = prepare_sequence(words, word_to_idx).unsqueeze(0).to(device)
        targets = prepare_sequence(tags, tag_to_idx).to(device)
        
        # Forward pass
        feats = model._get_lstm_features(sentence_in)
        feats = feats.squeeze(0)
        
        # Calculate loss
        loss = criterion(feats, targets)
        total_loss += loss.item()
        
        # Backward pass
        loss.backward()
        optimizer.step()
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/20, Loss: {total_loss:.4f}")

print("\nTraining completed!")

In [ ]:
# Test the model
def predict_tags(model, sentence, word_to_idx, idx_to_tag):
    model.eval()
    with torch.no_grad():
        sentence_in = prepare_sequence(sentence, word_to_idx).unsqueeze(0).to(device)
        _, tag_seq = model(sentence_in)
        tag_seq = tag_seq.squeeze(0).cpu().numpy()
        
        predicted_tags = [idx_to_tag[idx] for idx in tag_seq]
        return predicted_tags

# Test sentences
test_sentences = [
    ["Apple", "is", "in", "Cupertino"],
    ["Bill", "Gates", "founded", "Microsoft"],
]

print("\nPredictions:\n")
for sentence in test_sentences:
    predicted_tags = predict_tags(model, sentence, word_to_idx, idx_to_tag)
    
    print(f"Sentence: {' '.join(sentence)}")
    print(f"Tags:     {' '.join(predicted_tags)}")
    print()

## Part 5: Comparison and Evaluation

In [ ]:
# Compare all methods on the same text
comparison_text = "Elon Musk founded Tesla in California and SpaceX in Texas."

print("="*70)
print("COMPARISON OF NER METHODS")
print("="*70)
print(f"\nText: {comparison_text}\n")

# spaCy
print("\n1. spaCy Results:")
print("-" * 70)
doc_spacy = nlp(comparison_text)
for ent in doc_spacy.ents:
    print(f"  {ent.text:<20} -> {ent.label_}")

# NLTK
print("\n2. NLTK Results:")
print("-" * 70)
entities_nltk = extract_entities_nltk(comparison_text)
for entity, entity_type in entities_nltk:
    print(f"  {entity:<20} -> {entity_type}")

# Transformers
print("\n3. BERT Results:")
print("-" * 70)
results_bert = ner_pipeline(comparison_text)
for entity in results_bert:
    print(f"  {entity['word']:<20} -> {entity['entity_group']} (score: {entity['score']:.3f})")

print("\n" + "="*70)

## Summary

**Methods Covered:**
1. **spaCy**: Fast, accurate, easy to use for production
2. **NLTK**: Good for educational purposes, basic NER
3. **Transformers (BERT)**: State-of-the-art accuracy, slower
4. **Custom BiLSTM-CRF**: Full control, requires training data

**Entity Types:**
- PER (Person)
- ORG (Organization)
- LOC (Location)
- DATE (Date)
- MONEY (Monetary values)
- And more...

**Applications:**
- Information extraction
- Question answering
- Content classification
- Knowledge graph construction